# pii

> whether a document is somebody's business, decided by arithmetic rather than by a model

In [ ]:
#| default_exp pii

Checksummed patterns reach 0.996 precision at recall 1.000 (`evals/pii.py`). API keys
(`secret`) also gate. `ask(pii=…)` defines the modes: see [ask](02_ask.ipynb).


In [ ]:
#| export
import os, re
from fastcore.all import AttrDict, L


## What counts

- `MAX_SCAN`: sample both ends of a long document (headers hold account numbers).
- `DENSE`: matches per thousand chars. Callers can use it to separate a signature block from a customer list.
- `NER_CHARS` (20,000, from `extract`): cap on the opt-in name pass (`ner=True`).

In [ ]:
#| export
MAX_SCAN, DENSE = 200_000, 1.0

In [ ]:
#| export
def luhn(s:str) -> bool:
    "The check digit every payment card carries. Sixteen digits that fail it are not a card."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) < 12: return False
    tot, parity = 0, len(ds) % 2
    for i, d in enumerate(ds):
        if i % 2 == parity: d *= 2; d -= 9 if d > 9 else 0
        tot += d
    return tot % 10 == 0

def _iban_ok(s:str) -> bool:
    "IBAN's mod-97 check: move the country prefix to the end, letters to digits, remainder must be 1."
    s = re.sub(r'[^A-Za-z0-9]', '', s).upper()
    if not (15 <= len(s) <= 34): return False
    t = s[4:] + s[:4]
    try: n = int(''.join(str(int(c, 36)) for c in t))
    except ValueError: return False
    return n % 97 == 1

def _nhs_ok(s:str) -> bool:
    "The UK NHS number's mod-11 check digit. Ten digits in a row are otherwise just ten digits."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 10: return False
    tot = sum(d * (10 - i) for i, d in enumerate(ds[:9]))
    chk = 11 - tot % 11
    return chk != 10 and (0 if chk == 11 else chk) == ds[9]

def _ssn_ok(s:str) -> bool:
    "A US SSN's structurally impossible cases, which is as much as arithmetic can say about one."
    ds = re.sub(r'\D', '', s)
    if len(ds) != 9: return False
    a, b, c = ds[:3], ds[3:5], ds[5:]
    return a not in ('000', '666') and a[0] != '9' and b != '00' and c != '0000'

Checksums raise precision from 0.956 to 0.996 at recall 1.000 on 480 documents. False
positives fall from 11/240 to 1/240 (`evals/pii.py`). The remaining error is a Luhn-valid
random 16-digit run.


## What a number is not

A standards designation can have the shape of a US ZIP (`EN 60601`). A ten-digit path in a URL
segment passes the NHS mod-11 check about one time in eleven (`.../pages/2377744435`).
The surrounding text distinguishes both from PII.


In [ ]:
#| export
#: The fifty states plus DC. Two capitals before five digits are a ZIP only if they name a state.
US_STATES = frozenset(
    'AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM '
    'NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC'.split())
#: A city before the state, for the same reason the street line needs a street name: without it
#: `MS 73681` and `AL 31000` are addresses (held-out `stdbody` class, evals/RESULTS.md).
_ZIP = (r'\b[A-Z][a-z]+(?:[ ][A-Z][a-z]+){0,2},?[ ](?:' + '|'.join(sorted(US_STATES))
        + r') \d{5}(?:-\d{4})?\b')

#: What names the digits after it as a reference rather than as somebody's. Matched against the
#: text to the left of a match, so `EN 60601` and `Order 4556737586899855` never reach a checksum.
#: The acronyms stay case-sensitive: lowercase `en` is an ordinary word. `No` is not here:
#: it costs `No. 5 Elm Street` and buys no precision (evals/pii.py).
DESIGNATOR = re.compile(
    r'(?:(?-i:\b(?:ISO|IEC|EN|BS|DIN|JIS|NZS|ASTM|ANSI|IEEE|NIST|NFPA|MIL|STD|RFC|ISBN|ISSN|DOI'
    r'|PMID|PMCID|CVE|CWE|GTIN|EAN|UPC|SKU|MPN)\b)'
    r'|\b(?:arxiv|figure|fig|table|section|clause|annex|appendix|exhibit|schedule|paragraph|para'
    r'|item|step|page|pages|line|row|column|col|footnote|volume|vol|chapter|part|version|ver'
    r'|build|revision|rev|commit|sha|release|port|ticket|issue|bug|order|invoice|receipt|docket'
    r'|serial|batch|lot|tracking|shipment|transaction|grid reference|ref)\b)'
    r'[\s.:#=/_-]{0,4}$', re.I)

#: Kinds that match on length alone, and so lose a tie to any kind that matches on shape.
GENERIC = frozenset({'card', 'nhs', 'phone', 'account'})

#: Where the links are. Every digit run inside one is a path segment or a query value.
URLISH = re.compile(r'[a-z][a-z0-9+.-]*://\S+|\bwww\.\S+', re.I)
#: Kinds still worth finding inside a link: a key in a query string is still a key.
URL_KEEP = frozenset({'email', 'secret', 'ip'})

def _designated(s:str,  # the part being scanned
                i:int,  # where the match starts
) -> bool:
    "Is what starts at `i` introduced by a word that makes it a reference number?"
    return bool(DESIGNATOR.search(s[max(0, i - 48):i]))


### Europe, South and South East Asia, Australia

Nineteen regional identifiers, grouped by how much evidence the number carries on its own.

- Checksum: `nric` (Singapore), `nir` (France), `fnr` (Norway),
  `cf` (Italy), `dni`/`nie` (Spain), `abn` (Australia), `gstin` (India), `pan` (India).
  A mod-97 or mod-89 check, or a shape with four fixed letter positions, is not met by accident.
- Group or label: `aadhaar` (India), `thai_id`, `personnummer` (Sweden). Verhoeff, mod-11
  and Luhn each accept one bare digit run in ten. These form the `bare` lookalike class.
- Label: `tfn` and `medicare` (Australia), `bsn` (Netherlands), `pesel` (Poland),
  `steuerid` (Germany), `nino` (UK), `imei`. These have weak checks or none. The field label
  supplies the evidence.

Recall and precision per identifier in `evals/RESULTS.md`.


In [ ]:
#| export
#: Regional identifiers, one checksum each. Every function takes the matched text and ignores
#: separators, so `2234 5678 9012` and `223456789012` are the same Aadhaar. Sources for the
#: algorithms are the issuing authorities; the multilingual cue idea is from
#: `context_cued.py` in LiquidAI/LFM2.5-Encoder-350M-PII-Detector, which documents that a
#: shapeless ID has no learnable form and has to be gated by its field label.
# Verhoeff (Aadhaar)
_VD = [[0,1,2,3,4,5,6,7,8,9],[1,2,3,4,0,6,7,8,9,5],[2,3,4,0,1,7,8,9,5,6],[3,4,0,1,2,8,9,5,6,7],
       [4,0,1,2,3,9,5,6,7,8],[5,9,8,7,6,0,4,3,2,1],[6,5,9,8,7,1,0,4,3,2],[7,6,5,9,8,2,1,0,4,3],
       [8,7,6,5,9,3,2,1,0,4],[9,8,7,6,5,4,3,2,1,0]]
_VP = [[0,1,2,3,4,5,6,7,8,9],[1,5,7,6,2,8,3,0,9,4],[5,8,0,3,7,9,6,1,4,2],[8,9,1,6,0,4,3,5,2,7],
       [9,4,5,3,1,2,6,8,7,0],[4,2,8,6,5,7,3,9,0,1],[2,7,9,3,8,0,6,4,1,5],[7,0,4,6,9,1,3,2,5,8]]
_VINV = [0,4,3,2,1,5,6,7,8,9]

def verhoeff_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    c = 0
    for i, d in enumerate(reversed(ds)): c = _VD[c][_VP[i % 8][d]]
    return c == 0
def aadhaar_ok(s):
    ds = [c for c in s if c.isdigit()]
    return len(ds) == 12 and ds[0] not in '01' and verhoeff_ok(''.join(ds))

def gen_aadhaar(r):
    ds = [r.randrange(2, 10)] + [r.randrange(10) for _ in range(10)]
    return ''.join(map(str, ds + [verhoeff_digit(ds)]))

def tfn_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 9: return False
    return sum(d*w for d, w in zip(ds, (1,4,3,7,5,8,6,9,10))) % 11 == 0

def abn_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 11: return False
    ds = [ds[0] - 1] + ds[1:]
    return sum(d*w for d, w in zip(ds, (10,1,3,5,7,9,11,13,15,17,19))) % 89 == 0

def medicare_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) not in (10, 11) or not 2 <= ds[0] <= 6: return False
    return sum(d*w for d, w in zip(ds[:8], (1,3,7,9,1,3,7,9))) % 10 == ds[8]

_NRIC_T = {'S': 'JZIHGFEDCBA', 'T': 'JZIHGFEDCBA', 'F': 'XWUTRQPNMLK', 'G': 'XWUTRQPNMLK'}
_NRIC_OFF = {'S': 0, 'T': 4, 'F': 0, 'G': 4}

def nric_ok(s):
    s = s.upper().replace(' ', '')
    m = re.fullmatch(r'([STFG])(\d{7})([A-Z])', s)
    if not m: return False
    p, ds, chk = m.group(1), m.group(2), m.group(3)
    tot = sum(int(d)*w for d, w in zip(ds, (2,7,6,5,4,3,2))) + _NRIC_OFF[p]
    return _NRIC_T[p][tot % 11] == chk

def thai_id_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 13 or ds[0] == 0: return False
    return (11 - sum(d*(13-i) for i, d in enumerate(ds[:12])) % 11) % 10 == ds[12]

def bsn_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 9: return False
    return sum(d*w for d, w in zip(ds, (9,8,7,6,5,4,3,2,-1))) % 11 == 0

def nir_ok(s):
    s = re.sub(r'[^0-9A-Ba-b]', '', s).upper()
    if len(s) != 15: return False
    body = s[:13].replace('2A', '19').replace('2B', '18')
    if not body.isdigit(): return False
    try: return 97 - int(body) % 97 == int(s[13:])
    except ValueError: return False

_DNI_L = 'TRWAGMYFPDXBNJZSQVHLCKE'

def dni_ok(s):
    "DNI `12345678Z` and NIE `X1234567L`: the letter is the number mod 23, X/Y/Z standing for 0/1/2."
    s = s.upper().replace('-', '').replace(' ', '')
    if re.fullmatch(r'\d{8}[A-Z]', s): return _DNI_L[int(s[:8]) % 23] == s[8]
    m = re.fullmatch(r'([XYZ])(\d{7})([A-Z])', s)
    if not m: return False
    return _DNI_L[int(f'{"XYZ".index(m[1])}{m[2]}') % 23] == m[3]

_CF_ODD = {c: v for c, v in zip('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ',
    [1,0,5,7,9,13,15,17,19,21,1,0,5,7,9,13,15,17,19,21,2,4,18,20,11,3,6,8,12,14,16,10,22,25,24,23])}
_CF_EVEN = {c: (int(c) if c.isdigit() else ord(c)-65) for c in '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'}

def cf_ok(s):
    s = s.upper().replace(' ', '')
    if not re.fullmatch(r'[A-Z]{6}\d{2}[A-EHLMPR-T]\d{2}[A-Z]\d{3}[A-Z]', s): return False
    tot = sum((_CF_ODD if i % 2 == 0 else _CF_EVEN)[c] for i, c in enumerate(s[:15]))
    return chr(65 + tot % 26) == s[15]

def pesel_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 11: return False
    return (10 - sum(d*w for d, w in zip(ds[:10], (1,3,7,9,1,3,7,9,1,3))) % 10) % 10 == ds[10]

def _luhn_ds(ds):
    "Luhn over a digit list, for the national numbers that reuse it (Sweden, IMEI)."
    tot, par = 0, len(ds) % 2
    for i, d in enumerate(ds):
        if i % 2 == par: d = d*2 - 9 if d*2 > 9 else d*2
        tot += d
    return tot % 10 == 0

def personnummer_ok(s):
    "Luhn, and a real birth date. A Swedish organisationsnummer is Luhn over the same 6-4 shape."
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) == 12: ds = ds[2:]
    if len(ds) != 10 or not _luhn_ds(ds): return False
    mm, dd = ds[2]*10 + ds[3], ds[4]*10 + ds[5]
    return 1 <= mm <= 12 and 1 <= dd <= 31

def fnr_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 11: return False
    k1 = 11 - sum(d*w for d, w in zip(ds[:9], (3,7,6,1,8,9,4,5,2))) % 11
    k2 = 11 - sum(d*w for d, w in zip(ds[:10], (5,4,3,2,7,6,5,4,3,2))) % 11
    return (k1 % 11) == ds[9] and (k2 % 11) == ds[10]

def steuerid_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    if len(ds) != 11 or ds[0] == 0: return False
    p = 10
    for d in ds[:10]:
        m = (d + p) % 10 or 10
        p = (2*m) % 11
    return (11 - p) % 10 == ds[10]

def pan_ok(s):
    return bool(re.fullmatch(r'[A-Z]{3}[ABCFGHLJPTKE][A-Z]\d{4}[A-Z]', s.upper()))

_B36 = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def gstin_ok(s):
    s = s.upper()
    if not re.fullmatch(r'\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]', s): return False
    tot = 0
    for i, c in enumerate(s[:14]):
        v = _B36.index(c) * (2 if i % 2 else 1)
        tot += v // 36 + v % 36
    return _B36[(36 - tot % 36) % 36] == s[14]

_NINO_BAD = {'BG','GB','NK','KN','TN','NT','ZZ'}

def nino_ok(s):
    "Shape and prefix only: a NINO carries no check digit, which is why its pattern needs the cue."
    # the match includes the cue word, so read the candidate off the end
    m = re.search(r'[A-CEGHJ-PR-TW-Z][A-CEGHJ-NPR-TW-Z]\d{6}[A-D]?$',
                  re.sub(r'[^A-Za-z0-9]', '', s).upper())
    return bool(m) and m[0][:2] not in _NINO_BAD

def imei_ok(s):
    ds = [int(c) for c in s if c.isdigit()]
    return len(ds) == 15 and _luhn_ds(ds)


In [ ]:
#| export
#: kind -> (pattern, validator). Merged into `PATTERNS`, so `redact` writes `[AADHAAR]`.
INTL_PATTERNS = {
    # its own checksum is enough
    'nric':     (r'\b[STFG]\d{7}[A-Z]\b', nric_ok),
    'nir':      (r'\b[12][ ]?\d{2}[ ]?\d{2}[ ]?(?:\d{2}|2[AB])[ ]?\d{3}[ ]?\d{3}[ ]?\d{2}\b', nir_ok),
    'fnr':      (r'\b\d{6}[ ]?\d{5}\b', fnr_ok),
    'cf':       (r'\b[A-Z]{6}\d{2}[A-EHLMPR-T]\d{2}[A-Z]\d{3}[A-Z]\b', cf_ok),
    'dni':      (r'\b\d{8}[- ]?[A-Z]\b', dni_ok),
    'nie':      (r'\b[XYZ][- ]?\d{7}[- ]?[A-Z]\b', dni_ok),
    'abn':      (r'\b\d{2}[ ]?\d{3}[ ]?\d{3}[ ]?\d{3}\b', abn_ok),
    'gstin':    (r'\b\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]\b', gstin_ok),
    'pan':      (r'\b[A-Z]{3}[ABCFGHLJPTKE][A-Z]\d{4}[A-Z]\b', pan_ok),
    # grouped, or named: Verhoeff, mod-11 and Luhn each pass one bare run in ten
    'aadhaar':  (r'\b[2-9]\d{3}[ -]\d{4}[ -]\d{4}\b'
                 r'|\b(?:aadhaar|aadhar|uidai|आधार)\W{0,8}[2-9]\d{3}[ -]?\d{4}[ -]?\d{4}\b', aadhaar_ok),
    'thai_id':  (r'\b\d[ -]\d{4}[ -]\d{5}[ -]\d{2}[ -]\d\b'
                 r'|\b(?:thai\s*(?:national\s*)?id|บัตรประชาชน|เลขบัตร)\W{0,8}\d[ -]?\d{4}[ -]?\d{5}[ -]?\d{2}[ -]?\d\b', thai_id_ok),
    'personnummer': (r'\b\d{6}[-+]\d{4}\b'
                 r'|\b(?:personnummer|person\s*number)\W{0,8}(?:\d{2})?\d{6}[-+]?\d{4}\b', personnummer_ok),
    # named only: a weak check, or none at all
    'tfn':      (r'\b(?:tfn|tax\s*file\s*(?:no|number|#)?)\W{0,8}\d{3}[ ]?\d{3}[ ]?\d{3}\b', tfn_ok),
    'medicare': (r'\b(?:medicare(?:\s*(?:card|no|number|#))?)\W{0,8}\d{4}[ ]?\d{5}[ ]?\d(?:[ /]?\d)?\b', medicare_ok),
    'bsn':      (r'\b(?:bsn|burgerservicenummer|sofinummer)\W{0,8}\d{4}[ .]?\d{2}[ .]?\d{3}\b', bsn_ok),
    'pesel':    (r'\b(?:pesel)\W{0,8}\d{11}\b', pesel_ok),
    'steuerid': (r'\b(?:steuer\s*-?\s*id(?:nr|entifikationsnummer)?|idnr|steuerliche\s*identifikationsnummer)'
                 r'\W{0,8}\d{2}[ ]?\d{3}[ ]?\d{3}[ ]?\d{3}\b', steuerid_ok),
    'nino':     (r'\b(?:nino?|national\s*insurance(?:\s*(?:no|number|#))?)\W{0,8}'
                 r'[A-CEGHJ-PR-TW-Z][A-CEGHJ-NPR-TW-Z][ ]?\d{2}[ ]?\d{2}[ ]?\d{2}[ ]?[A-D]?\b', nino_ok),
    'imei':     (r'\b(?:imei|meid)\W{0,8}\d{2}[ -]?\d{6}[ -]?\d{6}[ -]?\d\b', imei_ok),
}
#: Case matters for the ones whose shape is letters in fixed positions.
INTL_CASED = frozenset({'nric', 'cf', 'dni', 'nie', 'gstin', 'pan'})


In [ ]:
#| export
#: kind -> (pattern, validator or None). Spans de-overlapped longest-first.
PATTERNS = {
    'email':   (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', None),
    # payment cards open 2-6 (Visa 4, Mastercard 2 and 5, Amex 3, Discover and UnionPay 6);
    # the other five leading digits are half of Luhn's collisions and none of its cards
    'card':    (r'\b[2-6](?:[ -]?\d){12,18}\b', luhn),
    'iban':    (r'\b[A-Z]{2}\d{2}[ ]?(?:[A-Z0-9]{4}[ ]?){2,7}[A-Z0-9]{1,4}\b', _iban_ok),
    'ssn':     (r'\b\d{3}-\d{2}-\d{4}\b', _ssn_ok),
    # grouped, or named: a bare ten-digit run passes mod-11 one time in eleven (evals/pii.py)
    'nhs':     (r'\b\d{3}[ -]\d{3}[ -]\d{4}\b'
                r'|\bnhs\s*(?:no|number|#)?\W{0,4}\d{3}[ -]?\d{3}[ -]?\d{4}\b', _nhs_ok),
    'phone':   (r'\+\d{1,3}[ .-]?\(?\d{1,5}\)?[ .-]?\d{3,4}[ .-]?\d{3,4}\b'
                r'|\(\d{2,5}\)[ .-]?\d{3,4}[ .-]?\d{3,4}\b'
                # not one link in a longer chain of groups: `0000-0002-1825-0097` is an ORCID
                r'|(?<![-\d.])\b0\d{1,4}[ .-]\d{3,4}[ .-]?\d{3,4}\b(?![-.]?\d)'
                r'|\b\d{3}-\d{3}-\d{4}\b'
                r'|\b(?:phone|tel|telephone|mobile|cell|fax)\b\W{0,8}\+?[\d ().-]{7,20}\d', None),
    'ip':      (r'\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b', None),
    'dob':     (r'\b(?:date of birth|dob|born)\b\W{0,12}(?:\d{1,4}[/-]\d{1,2}[/-]\d{1,4}|\d{1,2} \w+ \d{4})', None),
    'passport':(r'\b(?:passport(?:\s*(?:no|number|#))?)\W{0,6}[A-Z0-9]{6,9}\b', None),
    'licence': (r'\b(?:driver.?s? licen[cs]e|dl)(?:\s*(?:no|number|#))?\W{0,6}[A-Z0-9]{5,20}\b', None),
    'account': (r'\b(?:account|acct|a/c)(?:\s*(?:no|number|#))?\W{0,6}\d{6,17}\b', None),
    'sortcode':(r'\b(?:sort\s*code)\W{0,6}\d{2}[- ]?\d{2}[- ]?\d{2}\b', None),
    'secret':  (r'\b(?:sk-[A-Za-z0-9_-]{16,}|ghp_[A-Za-z0-9]{20,}|xox[baprs]-[A-Za-z0-9-]{10,}|AKIA[0-9A-Z]{16}|AIza[0-9A-Za-z_-]{35})\b', None),
    'medical': (r'\b(?:patient (?:id|number|name)|nhs number|medical record(?:\s*(?:no|number|#))?|mrn\W{0,6}\w+)\b', None),
    # require a street name between number and suffix, and a named state before the ZIP. The
    # ZIP list is worth 0.996 vs 0.949; the street name is covered by DESIGNATOR on the corpus
    # in evals/pii.py and worth 0.949 vs 0.833 with that guard switched off.
    'address': (r'\b\d{1,5}[A-Za-z]?[ ,]+(?:[A-Z][A-Za-z.\'-]+[ ,]+){1,3}'
                r'(?:Street|St|Road|Rd|Avenue|Ave|Lane|Ln|Drive|Boulevard|Blvd|Close|Court|Ct'
                r'|Crescent|Way|Place|Terrace|Square|Sq|Gardens|Grove|Row|Walk)\b\.?'
                r'|\b[A-Z]{1,2}\d[A-Z\d]? ?\d[A-Z]{2}\b'
                r'|' + _ZIP, None),
    **INTL_PATTERNS,
}
#: `person` is the one kind arithmetic cannot find, and the only one gated behind `ner=True`.
IDENTIFYING = frozenset({'email', 'card', 'iban', 'ssn', 'nhs', 'phone', 'dob', 'passport', 'licence', 'account',
                         'sortcode', 'medical', 'secret', 'address', 'person',
                         *INTL_PATTERNS})
CASED = frozenset({'address', *INTL_CASED})
_COMPILED = {k: (re.compile(p, 0 if k in CASED else re.I), v) for k, (p, v) in PATTERNS.items()}


## A detector fitted on your own examples

Arithmetic cannot know the identifiers one organisation invented. `EMP-483920` names somebody and
`ORD-483920` names a pallet, and no checksum separates them. `anya.pii` fits a tagger on a team's own
labelled examples; `use_learned` wires one in here, and from then on its kinds come back from
`pii_spans` beside the checksummed ones and count towards `has_pii`.

It speaks only for kinds `PATTERNS` does not have, so nothing above changes: the 0.996 precision is a
property of the patterns and wiring a model in does not touch them. `n` and `density` stay
arithmetic-only for the same reason they exclude `person`.

Its own baseline features come from `pattern_spans`, which is `pii_spans` with the learned layer off.
Anything else would have the detector ask itself what it thinks.


In [ ]:
#| export
#: A detector fitted by `anya.pii`, or None for arithmetic alone. `use_learned` sets it.
_LEARNED = None
_LOADED = {}

def _load_learned(model):
    "A fitted detector from a name, a path, or itself, with its baseline features wired to the patterns."
    if isinstance(model, str):
        if model in _LOADED: return _LOADED[model]
        try: from anya.pii import load_pii
        except ImportError: raise ImportError(
            "Fitting a PII detector lives in anya: pip install 'anya[pii]'") from None
        _LOADED[model] = det = load_pii(model)
    else: det = model
    det.base = pattern_spans          # or the features it reads would be its own output
    return det

def use_learned(model=None,   # a name or path `anya.pii.load_pii` takes, a fitted detector, or None to switch off
) -> object:
    "Wire a detector fitted on your own examples into every PII decision here. Returns it."
    global _LEARNED
    _LEARNED = _load_learned(model) if model is not None else None
    return _LEARNED

def learned(model=None,       # a detector, a name, None for whatever is wired, or False for none
) -> object:
    "The detector these functions consult: the argument, then `use_learned`, then `$VISHALAKSHI_PII_MODEL`."
    if model is False: return None
    if model is not None: return _load_learned(model)
    if _LEARNED is not None: return _LEARNED
    return use_learned(os.environ.get('VISHALAKSHI_PII_MODEL'))

def learned_spans(text:str, model=None) -> L:
    "What a fitted detector adds, as `(start, end, kind, text)`. Never a kind `PATTERNS` already has."
    det = learned(model)
    if det is None: return L()
    return L((s, e, k, v) for s, e, k, v in det.learned_spans(text) if k not in PATTERNS and k != 'person')

def identifying(model=None) -> frozenset:
    "The kinds that tip `has_pii`: `IDENTIFYING`, and whatever a fitted detector was taught."
    det = learned(model)
    return IDENTIFYING if det is None else IDENTIFYING | frozenset(det.kinds)


In [ ]:
#| export
def _scan_parts(text:str, mx:int=MAX_SCAN) -> list:
    """Both ends of a long document as `(offset, part)` (headers/footers hold identifiers)."""
    text = str(text or '')
    if len(text) <= mx: return [(0, text)]
    half = mx // 2
    return [(0, text[:half]), (half + 1, text[-half:])]

def _scan_text(text:str, mx:int=MAX_SCAN) -> str:
    "The part of a long document worth scanning, joined for measurement. Offsets follow `_scan_parts`."
    return '\n'.join(p for _, p in _scan_parts(text, mx))

def person_spans(text:str,     # what to scan
                 mx:int=None,  # chars handed to the extractor; None -> `extract.NER_CHARS`
) -> L:
    """Honorific-anchored personal names. Off by default; costs an entity pass."""
    from vishalakshi.extract import NER_CHARS, _noun_ents
    text = str(text or '')[:mx or NER_CHARS]
    out = []
    for surface, label in _noun_ents(text):
        if label != 'PERSON': continue
        # `_noun_ents` collapses whitespace, so match across the line breaks a PDF leaves inside
        # a name -- and on word boundaries, or `Ross` masks the middle of `Rossini`.
        rx = r'\b' + r'\s+'.join(map(re.escape, surface.split())) + r'\b'
        out += [(m.start(), m.end(), 'person', m.group(0)) for m in re.finditer(rx, text)]
    return L(out)

def pii_spans(text:str,          # what to scan
              kinds=None,        # restrict to these kinds; None -> every pattern
              mx:int=MAX_SCAN,   # chars scanned before a long document is sampled at both ends
              ner:bool=False,    # also look for names, which no pattern can find
              model=None,        # a fitted detector; None -> whatever `use_learned` wired, False -> none
) -> L:
    """Every match, as `(start, end, kind, text)`, longest first and never overlapping."""
    want = set(kinds or (*_COMPILED, 'person'))
    found = []
    for off, part in _scan_parts(text, mx):
        urls = [(m.start(), m.end()) for m in URLISH.finditer(part)]
        for kind, (rx, ok) in _COMPILED.items():
            if kind not in want: continue
            for m in rx.finditer(part):
                if ok is not None and not ok(m.group(0)): continue
                if kind not in URL_KEEP and any(a <= m.start() < b for a, b in urls): continue
                # only for a match that opens with a digit, so a cue-anchored one keeps its cue
                if m.group(0)[:1].isdigit() and _designated(part, m.start()): continue
                found.append((off + m.start(), off + m.end(), kind, m.group(0)))
        if ner and 'person' in want:
            found += [(off + s, off + e, k, v) for s, e, k, v in person_spans(part)]
        lsp = learned_spans(part, model)
        if kinds is not None: lsp = [s for s in lsp if s[2] in want]
        found += [(off + s, off + e, k, v) for s, e, k, v in lsp]
    # longest first, then leftmost, then specific over generic: `card` matches any Luhn-valid
    # 13-19 digit run, so a 15-digit `nir` or `imei` would otherwise be reported as a card
    found.sort(key=lambda s: (s[0] - s[1], s[0], s[2] in GENERIC))
    out, taken = [], []
    for s, e, kind, val in found:
        if any(s < te and ts < e for ts, te in taken): continue
        taken.append((s, e))
        out.append((s, e, kind, val))
    return L(sorted(out))

def pattern_spans(text:str, kinds=None, mx:int=MAX_SCAN, ner:bool=False) -> L:
    "`pii_spans` with the learned layer off: the patterns alone, which is what a fitted model is fed."
    return pii_spans(text, kinds, mx, ner, model=False)


In [ ]:
#| export
def pii_report(text:str,          # what to scan
               kinds=None,        # restrict to these kinds; None -> every pattern
               mx:int=MAX_SCAN,   # chars scanned before a long document is sampled at both ends
               ner:bool=False,    # also look for names
               model=None,        # a fitted detector; None -> whatever `use_learned` wired, False -> none
) -> AttrDict:
    """Spans found and whether they tip `has_pii` (IDENTIFYING kinds, and a fitted detector's own)."""
    spans, counts = pii_spans(text, kinds, mx, ner=ner, model=model), {}
    for _, _, k, _ in spans: counts[k] = counts.get(k, 0) + 1
    n = len(_scan_text(text, mx))
    ident = {k: v for k, v in counts.items() if k in identifying(model)}
    # `n` and `density` stay arithmetic-only, or `DENSE` would mean something new the day NER came on
    n_learned = sum(v for k, v in counts.items() if k not in PATTERNS and k != 'person')
    n_arith = len(spans) - counts.get('person', 0) - n_learned
    return AttrDict(has_pii=bool(ident), kinds=counts, identifying=ident, n=n_arith,
                    n_person=counts.get('person', 0), n_learned=n_learned, scanned=n,
                    scanned_ner=bool(ner), density=round(1000 * n_arith / max(n, 1), 3), spans=spans)


`has_pii` counts only `IDENTIFYING`. IP addresses remain reportable but do not trigger the gate.


In [ ]:
#| export
def redact(text:str,       # the text to mask
           spans=None,     # spans from `pii_spans`; recomputed over the whole of `text` when None
           kinds=None,     # restrict to these kinds
           mask:str=None,  # what to put in place of a match; None -> `[KIND]`
           ner:bool=False, # also mask names
           model=None,     # a fitted detector; None -> whatever `use_learned` wired, False -> none
) -> str:
    """Mask matched spans. Names only with `ner=True`."""
    out = str(text or '')
    if spans is None: spans = pii_spans(out, kinds, mx=len(out), ner=ner, model=model)
    for s, e, kind, _ in sorted(spans, reverse=True):
        out = out[:s] + (mask if mask is not None else f'[{kind.upper()}]') + out[e:]
    return out


## Asking the vault

`Vault.pii` is document-level. `pii_ctx` gates an answer over the sections retrieval chose.

`gated` gates retrieval itself. `ask` used to be the only method taking a policy. Every other one
handed raw section text to whatever read it next. `search`, `context`, `read`, `document`,
`sections`, `toc`, `doc_context` and `federate` all take `pii=` now. The default is `off`.

`redact` masks each identifier. `refuse` replaces the body. Either way the row stays, and `pii`
names the kinds. A caller that asked what matched still learns that something did.

## Learned models

The pretrained detectors do not improve the shipped gate. `evals/backends.py` holds the ONNX and
TFLite implementations, `evals/pii_model.py` measures them against arithmetic, and `evals/RESULTS.md`
records precision, recall, latency and blind spots.

A detector fitted on the corpus it will run over is the exception, and it is opt-in. `use_learned`
above wires one in; nothing is loaded and no kind changes until you do.


In [ ]:
#| export
from vishalakshi.core import Vault
from fastcore.all import patch

def redact_obj(o, kinds=None, ner:bool=False, model=None):
    "`redact` over the strings inside a nested dict or list: what a structured answer is."
    if isinstance(o, str):  return redact(o, kinds=kinds, ner=ner, model=model)
    if isinstance(o, dict): return {k: redact_obj(v, kinds, ner, model) for k, v in o.items()}
    if isinstance(o, list): return [redact_obj(v, kinds, ner, model) for v in o]
    return o

@patch
def pii(self:Vault,
        ref,                 # a doc_id, source, title or path: whatever `document` takes
        max_chars:int=MAX_SCAN,
        ner:bool=False,      # also look for names, except on code, where identifiers are not names
        model=None,          # a fitted detector; None -> whatever `use_learned` wired, False -> none
) -> AttrDict:
    "Whether one whole document is somebody's business, and what in it says so."
    d = self.document(ref, max_chars=max_chars)
    prose = (d.get('kind') or '') != 'code'
    # `scanned_ner` then reports False, which is the honest answer: nothing looked for a name here
    r = pii_report(d.text, ner=ner and prose, model=model)
    override = (self.marks(d.get('doc_id')) or {}).get('pii_override') if d.get('doc_id') else None
    r.detected, r.override = r.has_pii, override
    if override == 'clear': r.has_pii = False
    elif override == 'force': r.has_pii = True
    r.doc_id, r.title, r.source = d.get('doc_id'), d.get('title'), d.get('source')
    return r

@patch
def mark_not_pii(self:Vault, ref, clear:bool=True, reason:str='') -> dict:
    "Clear a false-positive PII decision (`pii_override='clear'`), or restore automatic detection."
    return self.mark(ref, pii_override='clear' if clear else None,
                     pii_reason=(reason or None) if clear else None)

@patch
def mark_pii(self:Vault, ref, force:bool=True, reason:str='') -> dict:
    "Force a document private even when arithmetic finds nothing (names, addresses, whole PDFs)."
    return self.mark(ref, pii_override='force' if force else None,
                     pii_reason=(reason or None) if force else None)

def pii_ctx(ctx, ner:bool=False, model=None) -> AttrDict:
    "The report for an assembled context, which is what a policy has to gate on."
    parts = [str(getattr(r, 'text', None) or (r.get('text') if isinstance(r, dict) else '') or '')
             for r in (list(ctx.get('results') or []) + list(ctx.get('related') or []))]
    return pii_report('\n\n'.join(parts), ner=ner, model=model)

In [ ]:
#| export
GATES = ('off', 'redact', 'refuse')
ROW_TEXT = ('text', 'snippet', 'snippets', 'content', 'summary')
ROW_LABEL = ('title', 'breadcrumb')
ROW_KIDS = ('tree', 'children')
CTX_ROWS = ('results', 'related', 'docs', 'hits')

def _pii_marks(v):
    "Cleared and force-private `(store, doc_id)` pairs, for whichever gate is asking."
    try: rows = list(v._marks()(where="pii_override IN ('clear','force')"))
    except Exception: return set(), set()
    return ({(r['store'], r['doc_id']) for r in rows if r['pii_override'] == 'clear'},
            {(r['store'], r['doc_id']) for r in rows if r['pii_override'] == 'force'})

def _row_get(r, k): return r.get(k) if isinstance(r, dict) else getattr(r, k, None)

def _row_doc(r):
    "The document a row belongs to. A row that names none falls back to its own text."
    # `search` and `context` say `doc_id`, `sections` only `node_id`, `read` says `id`, `federate` `ref`
    for k in ('doc_id', 'node_id', 'id', 'ref'):
        if v := _row_get(r, k): return str(v).split('#', 1)[0]

def _row_kinds(r, cleared, forced, store, ner=False):
    "What one row holds, honouring the mark on its document. `None` when it holds nothing."
    did = _row_doc(r)
    key = (_row_get(r, 'store') or store, did) if did else None
    if key and key in cleared: return None
    if key and key in forced: return {'marked': 1}
    parts = []
    for f in (*ROW_TEXT, *ROW_LABEL):
        v = _row_get(r, f)
        parts += [str(x) for x in v] if isinstance(v, (list, tuple)) else ([str(v)] if v else [])
    rep = pii_report('\n\n'.join(parts), ner=ner)
    return dict(rep.identifying) if rep.has_pii else None

def _section_private(r, cleared, forced, store:str, ner:bool=False) -> bool:
    "Whether one retrieved section is somebody's business. What `ask` filters its context on."
    return bool(_row_kinds(r, cleared, forced, store, ner))

def held(kinds) -> str:
    "What stands in for text a `refuse` policy will not hand back."
    return f"[withheld: personal information ({', '.join(sorted(kinds))})]"

def _mask(r, f, fix):
    "Rewrite field `f` of `r` through `fix`, whether it holds a string or a list of them."
    if not (v := r.get(f)): return
    r[f] = type(v)(fix(str(x)) for x in v) if isinstance(v, (list, tuple)) else fix(str(v))

def _gate_row(r, act, cleared, forced, store, ner):
    "One row masked or withheld, and whatever nests under it. A clean parent can hold a private child."
    if not isinstance(r, dict): return r
    kinds = _row_kinds(r, cleared, forced, store, ner)
    kids = {f: v for f in ROW_KIDS if isinstance(v := r.get(f), (dict, list, tuple, L))}
    if not kinds and not kids: return r
    out = type(r)(r)
    label = lambda s: redact(s, ner=ner)
    if kinds:
        for f in ROW_TEXT: _mask(out, f, label if act == 'redact' else lambda s: held(kinds))
        for f in ROW_LABEL: _mask(out, f, label)
        out['pii'] = kinds
    g = lambda x: _gate_row(x, act, cleared, forced, store, ner)
    for f, v in kids.items(): out[f] = g(v) if isinstance(v, dict) else type(v)(g(k) for k in v)
    return out

def gated(o,                  # whatever a retrieval primitive returned
          pii:str='off',      # off | redact | refuse
          vault=None,         # the shelf whose marks apply; None -> no marks
          ner:bool=False,     # gate on titled names too
          store:str=None,     # the shelf the rows came from; None -> the vault's own
):
    "Apply a retrieval policy to rows, an assembled context, or one section."
    if pii == 'off' or o is None: return o
    if pii not in GATES: raise ValueError(f'unknown pii policy for retrieval: {pii!r}; one of {GATES}')
    cleared, forced = _pii_marks(vault) if vault is not None else (set(), set())
    g = lambda r: _gate_row(r, pii, cleared, forced, store or getattr(vault, 'name', 'store'), ner)
    if isinstance(o, str):
        rep = pii_report(o, ner=ner)
        if not rep.has_pii: return o
        return redact(o, ner=ner) if pii == 'redact' else held(rep.identifying)
    if isinstance(o, (list, tuple, L)): return type(o)(g(r) for r in o)
    if not isinstance(o, dict): return o
    rows = [k for k in CTX_ROWS if isinstance(o.get(k), (list, tuple, L))]
    if not rows and not isinstance(o.get('doc'), dict): return g(o)
    out = type(o)(o)
    for k in rows: out[k] = L(g(r) for r in out[k])
    if isinstance(out.get('doc'), dict): out['doc'] = g(out['doc'])
    return out


In [ ]:
#| export
@patch
def toc(self:Vault,
        doc=None,            # narrow to one document, as `Index.toc` takes it
        pii:str='off',       # off | redact | refuse
        pii_ner:bool=False,  # gate on titled names too
        **kw                 # forwarded to `Index.toc`
) -> list:
    "The heading tree. Node titles are the openings of their sections, and a policy covers them too."
    from litesearch import Index
    return gated(Index.toc(self, doc, **kw), pii, self, ner=pii_ner)


In [ ]:
#| hide
from tempfile import mkdtemp
from pathlib import Path
from fastcore.test import test_eq, test_fail
from vishalakshi import Vault

_g = Vault(str(Path(mkdtemp())/'gate.db'), offline=True)
_g.note('Invoice 4471 for Ada, ada@example.com, phone 020 7946 0958. Card 4111 1111 1111 1111.',
        title='invoice 4471')
_g.note('The deploy pipeline runs on GitHub Actions and takes 20 minutes.', title='pipeline')
_inv, _pipe = _g.doc('invoice 4471')['id'], _g.doc('pipeline')['id']
_LEAK = ('ada@example.com', '4111 1111 1111 1111', '020 7946 0958')

def _leaks(o): return [s for s in _LEAK if s in str(o)]

#: every primitive that hands section text back, and how to call it on this vault
_prims = dict(
    search      = lambda **k: _g.search('invoice 4471', **k),
    sections    = lambda **k: _g.sections('invoice 4471', **k),
    context     = lambda **k: _g.context('invoice 4471', code=0, shelves=0, **k),
    read        = lambda **k: _g.read(f'{_inv}#0', **k),
    document    = lambda **k: _g.document(_inv, **k),
    doc_context = lambda **k: _g.doc_context(_inv, 'invoice', related=0, **k),
)

# `off` is the default: a caller that does not ask gets what it always got
for nm, f in _prims.items(): test_eq((nm, bool(_leaks(f()))), (nm, True))
# ...and neither mode leaves an identifier behind
for act in ('redact', 'refuse'):
    for nm, f in _prims.items(): test_eq((act, nm, _leaks(f(pii=act))), (act, nm, []))

# `local` is `ask`'s: retrieval has no model to send a question to
test_fail(lambda: _g.read(f'{_inv}#0', pii='local'), contains='off')
test_fail(lambda: _g.read(f'{_inv}#0', pii='sortof'), contains='unknown pii policy')

In [ ]:
#| hide
# What the two modes do to one row, and what they leave alone
_row = _g.read(f'{_inv}#0', pii='refuse')
test_eq(_row['text'], '[withheld: personal information (card, email, phone)]')
# the kinds, so a caller can say why. Counted over body and label together, so a note whose title
# repeats its first line reports each kind twice; the kinds are the answer, the tallies are not
test_eq(sorted(_row['pii']), ['card', 'email', 'phone'])
test_eq(_row['title'], 'invoice 4471')                         # a clean label is left as it is
test_eq('[EMAIL]' in _g.read(f'{_inv}#0', pii='redact')['text'], True)

# a title that carries an identifier is masked rather than withheld, under either mode: a note is
# titled by its own first line, and a row with no label left cannot be opened or asked about
_titled = dict(doc_id='d1', title='Invoice for ada@example.com', text='Invoice for ada@example.com')
for _act in ('redact', 'refuse'):
    _t = gated([_titled], _act)[0]
    test_eq(_t['title'], 'Invoice for [EMAIL]')
    test_eq(_leaks(_t), [])
test_eq(gated([_titled], 'refuse')[0]['text'], '[withheld: personal information (email)]')

# `summary` is a sentence of the section, so it holds whatever the section holds
for _f in ('text', 'summary'): assert not _leaks(_row.get(_f)), _f

# a clean row is returned untouched, and carries no verdict
_clean = [r for r in _g.search('deploy pipeline', pii='refuse') if 'GitHub' in str(r['snippet'])]
test_eq(len(_clean), 1)
test_eq('pii' in _clean[0], False)

# the marks reach every primitive, not just `ask`
_g.mark_not_pii(_inv, reason='my own invoice')
for nm, f in _prims.items(): test_eq((nm, bool(_leaks(f(pii='refuse')))), (nm, True))
_g.mark_not_pii(_inv, clear=False)
test_eq(_leaks(_g.read(f'{_inv}#0', pii='refuse')), [])

# ...including on a document arithmetic finds nothing in
test_eq('GitHub' in str(_g.read(f'{_pipe}#0', pii='refuse')), True)
_g.mark_pii(_pipe, reason='confidential')
_forced = _g.read(f'{_pipe}#0', pii='refuse')
test_eq(('GitHub' in str(_forced), _forced['pii']), (False, {'marked': 1}))
_g.mark_pii(_pipe, force=False)

# a row whose document cannot be named is still gated, on its text alone
test_eq(_leaks(gated([dict(text=_LEAK[0])], 'refuse')), [])
test_eq(gated('nothing identifying', 'refuse'), 'nothing identifying')
test_eq(gated(None, 'refuse'), None)

In [ ]:
report = pii_report("""
Invoice 4471 for Ada Lovelace <ada@example.com>, phone 020 7946 0958.
Card 4111 1111 1111 1111, sort code 20-00-00, account number 12345678.
Server 10.0.0.14 returned 500. Order number 4471000012345678.
""")
report.has_pii, report.identifying, report.n

(True, {'email': 1, 'phone': 1, 'card': 1, 'sortcode': 1, 'account': 1}, 6)

In [ ]:
from fastcore.test import test_eq
test_eq(report.has_pii, True)
test_eq('card' in report.identifying, True)      # passes Luhn
test_eq(report.kinds.get('ip'), 1)               # reported...
test_eq('ip' in report.identifying, False)       # ...but a log is not somebody's private life

In [ ]:
print(redact("""
Invoice 4471 for Ada Lovelace <ada@example.com>, phone 020 7946 0958.
Card 4111 1111 1111 1111, sort code 20-00-00.
""").strip())

Invoice 4471 for Ada Lovelace <[EMAIL]>, [PHONE].
Card [CARD], [SORTCODE].


Names are opt-in and honorific-anchored. `Dr Charles Babbage` matches. Bare `Ada Lovelace`
does not. Read `scanned_ner` before interpreting a zero. Street lines use patterns.

The ONNX detector finds 5/8 blind-spot examples against 2/8 for `ner=True`, but costs one
gigabyte of weights. It does not ship. `evals/backends.py` contains both learned detectors.


In [ ]:
#| hide
from fastcore.test import test_eq

# secrets gate: a key alone is identifying
test_eq(pii_report('export OPENAI_API_KEY=sk-abcdefghijklmnopqrstuvwxyz123456').has_pii, True)
test_eq('secret' in pii_report('token ghp_abcdefghijklmnopqrst').identifying, True)

# medical: research prose is not a medical record; a patient id is
test_eq(pii_report('This paper diagnoses a failure mode in the prescription of learning rates').has_pii, False)
test_eq(pii_report('Patient id 44291 was discharged yesterday.').has_pii, True)

# nested redact for structured answers
test_eq(redact_obj({'email': 'a@b.co', 'n': 1}), {'email': '[EMAIL]', 'n': 1})
test_eq(redact_obj(['a@b.co', 3])[0], '[EMAIL]')


# a street line is what makes "John Smith, 12 Elm Street" private: no pattern finds the name
test_eq(pii_report('John Smith, 12 Elm Street').identifying, {'address': 1})
test_eq(pii_report('900 Market St, San Francisco CA 94103').has_pii, True)
test_eq(pii_report('The registered office is 221B Baker Street, London NW1 6XE').kinds['address'], 2)
# a numbered heading is not an address, which is the whole precision argument
for _t in ('Chapter 4 Court decisions', 'Table 3 Road traffic figures', 'Figure 2 Way of working'):
    test_eq(pii_report(_t).has_pii, False)

In [ ]:
#| hide
# 1. off by default, capped at NER_CHARS when on
from vishalakshi.extract import NER_CHARS
_sig = 'Dr Charles Babbage signed it.'
test_eq(pii_report(_sig).has_pii, False)                       # names are not looked for
test_eq(pii_report(_sig, ner=True).identifying, {'person': 1})   # ...until asked
test_eq(pii_report('Ada Lovelace signed it.', ner=True).has_pii, False)   # an honorific is the anchor
test_eq(pii_report('x'*NER_CHARS + ' ' + _sig, ner=True).kinds.get('person'), None)   # past the cap

# 2. `scanned_ner` keeps "none found" apart from "not looked for"
test_eq(pii_report('nothing here').scanned_ner, False)
test_eq(pii_report('nothing here', ner=True).scanned_ner, True)

# names are masked once asked for, and not before
test_eq('[PERSON]' in redact(_sig), False)
test_eq(redact(_sig, ner=True), 'Dr [PERSON] signed it.')

# the seam: `_scan_parts` keeps the halves apart, so an honorific at the end of one and a
# capitalised pair at the start of the next is not a person who was never in the document
_half = 4000
_doc = 'x'*(_half-3) + ' Dr' + 'm'*5000 + 'Charles Babbage wrote it.' + 'z'*(_half-25)
test_eq(pii_report(_doc, mx=8000, ner=True).kinds.get('person'), None)

# `density` and `n` stay arithmetic-only, or `DENSE` would mean something new the day NER came on
_names = 'Dr Ada Lovelace met Dr Charles Babbage. '*5
test_eq((pii_report(_names).density, pii_report(_names, ner=True).density), (0.0, 0.0))
test_eq(pii_report(_names, ner=True).n_person, 10)

In [ ]:
#| hide
from tempfile import mkdtemp
from pathlib import Path
from vishalakshi import Vault

v = Vault(Path(mkdtemp())/'p.db', offline=True)
v.add('A letter about Jane, and what she said on Tuesday.', title='letter', source='/inbox/letter.md')
r = v.pii('/inbox/letter.md')
test_eq(r.has_pii, False)
v.mark_pii('/inbox/letter.md', reason='address book')
test_eq(v.pii('/inbox/letter.md').has_pii, True)
test_eq(v.pii('/inbox/letter.md').override, 'force')
v.mark_not_pii('/inbox/letter.md', reason='my own draft')
test_eq(v.pii('/inbox/letter.md').has_pii, False)
test_eq(v.pii('/inbox/letter.md').override, 'clear')

In [ ]:
#| hide
# 3. no NER on code: an identifier is not a name, and a report must not claim it looked
v.add('# Dr Charles Babbage wrote this\ndef f(): pass', title='mod', source='/m.py', kind='code')
_c = v.pii('/m.py', ner=True)
test_eq((_c.scanned_ner, _c.has_pii), (False, False))

v.add(_sig, title='signed', source='/inbox/signed.md')
_p = v.pii('/inbox/signed.md', ner=True)
test_eq((_p.scanned_ner, _p.has_pii, _p.identifying), (True, True, {'person': 1}))
test_eq(v.pii('/inbox/signed.md').scanned_ner, False)   # the default is still arithmetic only

A number that fails its checksum is not the thing the checksum protects.

In [ ]:
test_eq(pii_report('Order 4111 1111 1111 1112 shipped').has_pii, False)   # fails Luhn
test_eq(pii_report('Card 4111 1111 1111 1111 charged').has_pii, True)     # passes it
test_eq(pii_report('The build takes 20 minutes and costs nothing.').has_pii, False)

In [ ]:
#| hide
long_doc = 'Account number 12345678\n' + ('filler text. ' * 40_000) + '\nsigned, ada@example.com'
r = pii_report(long_doc)
test_eq(r.has_pii, True)
test_eq(sorted(r.identifying), ['account', 'email'])
test_eq(r.scanned <= MAX_SCAN + 1, True)

# ...but a *report* may sample and a redaction may not
masked = redact(long_doc)
assert 'ada@example.com' not in masked, masked[-80:]
assert '12345678' not in masked, masked[:80]
test_eq(masked.count('filler text. '), 40_000)      # and nothing in between was moved
test_eq(pii_report(masked).has_pii, False)

In [ ]:
#| hide
one = pii_report('4111 1111 1111 1111')
test_eq(one.n, 1)
test_eq(list(one.kinds), ['card'])

Phones need a separator or trunk prefix. Cards need an issuer digit. Ten-digit NHS numbers need
groups or a label. Each constraint raises precision from 0.992 to 0.996. Bare digit runs are not
identity.


In [ ]:
#| hide
# The gate is arithmetic until somebody wires a model in, which is what the `model` parameter has to
# be worth. It defaults to None everywhere it appears, `learned()` finds nothing to load, and every
# one of these answers exactly what the patterns answer.
import inspect

for _fn in (pii_spans, pii_report, redact, redact_obj, Vault.pii, pii_ctx):
    test_eq(inspect.signature(_fn).parameters['model'].default, None)
test_eq((learned(), _LEARNED, _LOADED), (None, None, {}))
_arith = 'Card 4111 1111 1111 1111 for ada@example.com, staff EMP-483920, order ORD-483920.'
test_eq(pii_spans(_arith), pattern_spans(_arith))
test_eq(sorted(pii_report(_arith).kinds), ['card', 'email'])         # EMP- is nobody's pattern
test_eq(pii_report(_arith).n_learned, 0)
test_eq(pii_report(_arith).n, pii_report(_arith, model=False).n)


In [ ]:
#| hide
# And what it is worth once one is wired. Needs `anya[pii]`, which is an extra, so this is skipped
# rather than failed where it is not installed.
try:
    from anya.pii import fit
    _fitted = fit([{'text': t, 'spans': [[s, s + len(v), 'emp_id']]}
                   for t, s, v in [(f'Approved by EMP-{n:06d} on the day.', 12, f'EMP-{n:06d}')
                                   for n in range(100000, 100040)]]
                  + [{'text': f'Order ORD-{n:06d} shipped from Leeds.', 'spans': []}
                     for n in range(100000, 100040)], base=pattern_spans)
except ImportError:
    _fitted = None

if _fitted is not None:
    _t = 'Approved by EMP-774310, against order ORD-774310, card 4111 1111 1111 1111.'
    test_eq(learned_spans(_t, _fitted).map(lambda s: s[2]), ['emp_id'])   # and not the order number
    _r = pii_report(_t, model=_fitted)
    test_eq(_r.has_pii, True)
    test_eq(sorted(_r.kinds), ['card', 'emp_id'])
    test_eq((_r.n, _r.n_learned), (1, 1))                    # `density` stays arithmetic-only
    test_eq('emp_id' in _r.identifying, True)                # a learned kind tips the gate
    test_eq(redact(_t, model=_fitted).count('[EMP_ID]'), 1)
    # a kind the patterns own is never the model's to speak for, so the checksummed layer is untouched
    test_eq(learned_spans('card 4111 1111 1111 1111', _fitted), [])
    test_eq(pii_report(_t, model=False).kinds, pii_report(_t).kinds)
    # `use_learned` wires one in for every call, including the ones with no `model` parameter at all
    use_learned(_fitted)
    test_eq(pii_report(_t).has_pii, True)
    test_eq('emp_id' in pii_report(_t).kinds, True)
    test_eq(gated({'results': [{'doc_id': 'd', 'text': _t}]}, 'redact')['results'][0]['pii'],
            {'card': 1, 'emp_id': 1})
    use_learned(None)
    test_eq((learned(), 'emp_id' in pii_report(_t).kinds), (None, False))


In [ ]:
#| hide
# What must and must not put a document on the local-only path
cases = [
    ('Order 4111 1111 1111 1112 shipped',                 False),   # fails Luhn
    ('Card 4111 1111 1111 1111 charged',                  True),
    ('phone 020 7946 0958',                               True),
    ('+44 20 7946 0958',                                  True),
    ('call (555) 123-4567',                               True),
    ('555-123-4567',                                      True),
    ('ada@example.com',                                   True),
    ('The build takes 20 minutes and costs nothing.',     False),
    ('Run 2024 1000 2000 3000 through the pipeline',      False),
    ('Release 1.2.3 shipped on 2024-05-01 with 400 tests', False),
    ('Server 10.0.0.14 returned 500',                     False),   # reported, not identifying
    ('commit 8f3a2b1 touched 120 lines in 14 files',      False),
]
for text, want in cases: test_eq((text, pii_report(text).has_pii), (text, want))

In [ ]:
#| hide
# The two failures this corpus was extended for. A standards designation has the shape of a US
# ZIP, and a ten-digit path segment passes the NHS mod-11 check about one time in eleven.
for _t in ('EN 60601-1 applies to medical electrical equipment.',
           'EN 60601-1-2 covers electromagnetic compatibility.',
           'Conformity with EN 60601 is required.',
           'BS EN 12345 was withdrawn.',
           'See https://example.atlassian.net/wiki/spaces/TA/pages/2377744435 for the spec.',
           'Order 2377744435 shipped.',
           'Page 2377744435 of the export was truncated.'):
    test_eq((_t, pii_report(_t).has_pii), (_t, False))

# ...without giving up the thing each guard sits next to
# the street line and the state+ZIP are two spans, and the ZIP needs the state to be one at all
test_eq(pii_report('Ship to 900 Market St, San Francisco CA 94103.').kinds, {'address': 2})
test_eq(pii_report('Ship to 900 Market St, San Francisco EN 94103.').kinds, {'address': 1})
test_eq(pii_report('NHS number 4505577104 recorded at triage.').has_pii, True)
test_eq(pii_report('Recorded at triage as 450 557 7104 on the day.').has_pii, True)
test_eq(pii_report('No. 5 Elm Street, London.').has_pii, True)   # `No.` numbers a house too
# a key in a query string is still a key, and an address in one is still an address
test_eq('secret' in pii_report('https://api.example.com/k?t=sk-abcdefghijklmnopqrstuvwxyz123456').identifying, True)
test_eq('email' in pii_report('https://x.example.com/f?to=jane@example.com').identifying, True)


In [ ]:
#| hide
# the regional identifiers, one per region, each rejected when its checksum is broken
_intl = [
    ('aadhaar',      'Aadhaar 2345 6789 0124 on file.',            'Aadhaar 2345 6789 0125 on file.'),
    ('pan',          'PAN ABCPE1234F quoted.',                     'PAN ABCXE1234F quoted.'),
    ('nric',         'NRIC S1234567D issued.',                      'NRIC S1234567A issued.'),
    ('abn',          'ABN 51 824 753 556 registered.',              'ABN 51 824 753 557 registered.'),
    ('tfn',          'TFN 123 456 782 lodged.',                     'TFN 123 456 783 lodged.'),
    ('dni',          'DNI 12345678Z presented.',                    'DNI 12345678A presented.'),
    ('imei',         'IMEI 49 015420 323751 8 blocked.',            'IMEI 49 015420 323751 9 blocked.'),
]
for _k, _good, _bad in _intl:
    test_eq((_k, _k in pii_report(_good).kinds), (_k, True))
    test_eq((_k, _k in pii_report(_bad).kinds), (_k, False))

# a Swedish company shares the personnummer format and its Luhn check; only the date separates them
test_eq('personnummer' in pii_report('Personnummer 850101-2382 noted.').kinds, True)
test_eq(pii_report('Organisationsnummer 556036-0793 for the entity.').has_pii, False)
# a specific kind beats a generic one on an identical span
test_eq(list(pii_spans('Numero de securite sociale 144017454301571.'))[0][2], 'nir')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()